# AD Qurro Metagenomics

In [1]:
import qiime2
import biom
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
from qiime2.plugins import feature_table
from qiime2 import Artifact
from skbio.stats import subsample_counts

In [10]:
#metadata
md = pd.read_csv('metagenomics/stool.metagenomic.metadata.with_amyloid_saczt.tsv', sep='\t', index_col=0).reset_index()
md['sample_name'] = md['sample_name'].str.replace(r'^\d+\.', '', regex=True)
sorted(list(md.columns))

['TRF_start_age_mo',
 'TRF_start_date',
 'age_of_collection_days',
 'age_of_collection_months',
 'box',
 'brain_insoluble_ab40',
 'brain_insoluble_ab42',
 'brain_lysate_ab40',
 'brain_lysate_ab42',
 'brain_soluble_ab40',
 'brain_soluble_ab42',
 'cage_id',
 'collection_timepoint',
 'collection_timestamp',
 'dab_area_cortex',
 'dab_area_hippocampus',
 'dab_count_cortex',
 'dab_count_hippocampus',
 'date_of_collection',
 'description',
 'dna_extracted',
 'dob',
 'elevation',
 'empo_1',
 'empo_2',
 'empo_3',
 'env_biome',
 'env_feature',
 'env_material',
 'env_package',
 'feeding_condition',
 'genotype_diet',
 'geo_loc_name',
 'host_age',
 'host_age_units',
 'host_body_habitat',
 'host_body_product',
 'host_body_site',
 'host_common_name',
 'host_scientific_name',
 'host_subject_id',
 'host_taxid',
 'host_weight',
 'host_weight_units',
 'iacuc_institute',
 'iacuc_protocol_id',
 'katharoseq_cell_count',
 'katharoseq_community_type',
 'katharoseq_control_type',
 'katharoseq_plate_number',
 '

In [7]:
md

,title,empo_1,empo_2,empo_3,latitude,taxon_id,elevation,env_biome,longitude,description,...,brain_lysate_ab40,brain_lysate_ab42,brain_insoluble_ab40,brain_insoluble_ab42,brain_soluble_ab40,brain_soluble_ab42,genotype_diet,zt_time,sac_ZT,sac_stool match
sample_name,,,,,,,,,,,,,,,,,,,,,
14708.301.T0.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ntg.ALF,0,NaN,NaN
14708.301.T12.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ntg.ALF,12,NaN,NaN
14708.301.T16.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ntg.ALF,16,NaN,NaN
14708.301.T20.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ntg.ALF,20,NaN,NaN
14708.301.T4.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Ntg.ALF,4,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14708.335.T12.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Tg.ALF,12,NaN,NaN
14708.335.T16.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Tg.ALF,16,NaN,NaN
14708.335.T20.S,ALZ/TRF,Host-associated,Animal,Animal proximal gut,32.5,410661,193,urban biome,-117.25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Tg.ALF,20,NaN,NaN


In [3]:
#load genome counts
genome_counts = Artifact.load('metagenomics/159013_wolR2_per_genome.feature-table.counts.de_qiitaed.no_blanks.qza').view(pd.DataFrame)

In [5]:
genome_counts.T.to_csv('metagenomics/159013_wolR2_per_genome.feature-table.counts.de_qiitaed.no_blanks.tsv', sep='\t')

In [8]:
genome_counts

,G000005825,G000006725,G000006785,G000006865,G000006925,G000007145,G000007265,G000007385,G000007465,G000007525,...,G902799635,G902800075,G902800385,G902801125,G902801355,G902801795,G902801825,G902801975,G902809815,G902825795
328.T4.S,6.0,0.0,1.0,9.0,20.0,1.0,4.0,1.0,1.0,2.0,...,2.0,2.0,4.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
315.T4.S,1.0,0.0,0.0,3.0,163.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,14.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
322.T20.S,0.0,0.0,0.0,2.0,212.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,17.0,0.0,1.0,0.0,3.0,0.0,0.0
335.T16.S,1.0,0.0,0.0,8.0,15.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,12.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
322.T0.S,0.0,0.0,0.0,0.0,17.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334.T16.S,0.0,0.0,0.0,2.0,65.0,0.0,0.0,0.0,0.0,0.0,...,2.0,1.0,5.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
334.T20.S,0.0,0.0,0.0,0.0,16.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
334.T4.S,0.0,0.0,0.0,3.0,39.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
334.T8.S,0.0,1.0,0.0,0.0,8.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
qiime diversity beta \
--i-table 159171_rs210_per_genome.feature-table.qza \
--p-metric braycurtis \
--o-distance-matrix AD.stool.metagenomic.braycurtis.distancematrix.qza

qiime diversity pcoa-biplot \
--i-features 159171_rs210_per_genome.feature-table.qza \
--i-pcoa bray_curtis_pcoa_results.qza \
--o-biplot AD.stool.metagenomic.braycurtis.biplot.qza



In [ ]:
qiime qurro loading-plot \
    --i-table for_qurro.counts.qza \
    --i-ranks wol2_genome_none.gemelli.biplot.qza \
    --m-sample-metadata-file overly_specified_metadata.tsv \
    --m-feature-metadata-file lineages.split.tsv \
    --o-visualization AD.stool.metagenomic.gemelli.qurro.qzv